In [3]:
%pip install pandas numpy scikit-learn lightgbm streamlit pydeck shap joblib geopy

  Using cached pandas-3.1.0-cp315-cp315-win_amd64.whl.metadata (18 kB)
  Using cached numpy-2.5.4-cp315-cp315-win_amd64.whl.metadata (6.6 kB)
  Using cached scikit_learn-1.9.1-cp315-cp315-win_amd64.whl.metadata (9.3 kB)
  Using cached lightgbm-4.7.0-py3-none-win_amd64.whl.metadata (18 kB)
  Using cached streamlit-1.66.1-py3-none-any.whl.metadata (10 kB)
  Using cached pydeck-0.9.3-py2.py3-none-any.whl.metadata (4.2 kB)
  Using cached shap-0.54.1-cp312-abi3-win_amd64.whl.metadata (23 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached geopy-2.5.0-py3-none-any.whl.metadata (7.1 kB)
  Using cached scipy-1.18.1-cp315-cp315-win_amd64.whl.metadata (61 kB)
  Using cached narwhals-2.28.0-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.7.0-py3-none-any.whl.metadata (24 kB)
  Using cached altair-6.3.0-py3-none-any.whl.metadata (11 kB)
  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached pillow-13.0.0-cp315-cp315-win_amd64.whl

In [6]:
import pandas as pd
import numpy as np

# 1. Load dataset using a raw string path
df = pd.read_csv(r'C:\Users\dhruv\OneDrive\Desktop\mumbai_real_estate.csv')

# 2. Distance function to major Mumbai business hubs (Vectorized Haversine)
def haversine_distance(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    return 6367 * 2 * np.arcsin(np.sqrt(a))

# Proximity to major commercial centers
df['dist_bkc_km'] = haversine_distance(df['latitude'], df['longitude'], 19.0657, 72.8686)      # BKC
df['dist_nariman_km'] = haversine_distance(df['latitude'], df['longitude'], 18.9256, 72.8242)  # Nariman Point
df['dist_airport_km'] = haversine_distance(df['latitude'], df['longitude'], 19.0896, 72.8656)  # Airport

# 3. Categorical encoding
df['furnished_score'] = df['furnished'].map({'Unfurnished': 0, 'Semi-Furnished': 1, 'Furnished': 2})

# 4. Target variable engineering: Yield % & Monthly Rent
base_yield = 3.0
yield_adj = (
    0.3 * (df['furnished_score'] == 2) +
    0.15 * (df['furnished_score'] == 1) +
    0.3 * (df['dist_bkc_km'] < 10) +
    0.2 * (df['age'] <= 5) -
    0.2 * (df['age'] > 15)
)
df['rental_yield_pct'] = np.clip(base_yield + yield_adj, 2.0, 5.0)
df['estimated_monthly_rent'] = np.round((df['price'] * (df['rental_yield_pct'] / 100)) / 12, 0)

print(f"Processed {len(df):,} properties successfully.")
display(df[['locality', 'price', 'area', 'bedroom_num', 'dist_bkc_km', 'rental_yield_pct', 'estimated_monthly_rent']].head())

Processed 71,938 properties successfully.


,locality,price,area,bedroom_num,dist_bkc_km,rental_yield_pct,estimated_monthly_rent
0,Kalyan,6600283,757,2,33.301056,3.2,17601.0
1,Kalyan,6169841,652,2,36.315593,3.2,16453.0
2,Dombivali,4599936,396,1,27.426130,3.2,12266.0
3,Ville Parle,51980000,1130,3,4.014010,3.5,151608.0
4,Nala Sopara,3915000,435,1,39.925815,3.2,10440.0


In [7]:
from sklearn.model_selection import train_test_split
import lightgbm as lgb
import joblib

# 1. Select the features the model will learn from
features = ['area', 'bedroom_num', 'bathroom_num', 'age', 'dist_bkc_km', 'dist_nariman_km', 'dist_airport_km', 'furnished_score']
X = df[features]
y = df['estimated_monthly_rent']

# 2. Split data: 80% for the model to study (train), 20% to test it
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Train the LightGBM Regressor
print("Training the LightGBM AI model. This might take a few seconds...")
model = lgb.LGBMRegressor(n_estimators=150, learning_rate=0.1, random_state=42)
model.fit(X_train, y_train)

# 4. Check how well it learned
accuracy = model.score(X_test, y_test)
print(f"Model Training Complete! Accuracy (R-squared score): {accuracy:.2f}")

# 5. Save the trained model into your 'models' folder for the Streamlit app
joblib.dump(model, '../models/mumbai_rent_model.pkl')
print("✅ Model successfully saved as 'mumbai_rent_model.pkl'!")

Training the LightGBM AI model. This might take a few seconds...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001367 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1091
[LightGBM] [Info] Number of data points in the train set: 57550, number of used features: 8
[LightGBM] [Info] Start training from score 54811.247263
Model Training Complete! Accuracy (R-squared score): 0.84
✅ Model successfully saved as 'mumbai_rent_model.pkl'!


In [8]:

%pip install matplotlib

   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.5 MB 3.8 MB/s eta 0:00:03
   ------ --------------------------------- 1.6/9.5 MB 3.7 MB/s eta 0:00:03
   -------- ------------------------------- 2.1/9.5 MB 3.4 MB/s eta 0:00:03
   ------------ --------------------------- 2.9/9.5 MB 3.5 MB/s eta 0:00:02
   --------------- ------------------------ 3.7/9.5 MB 3.6 MB/s eta 0:00:02
   ------------------ --------------------- 4.5/9.5 MB 3.6 MB/s eta 0:00:02
   ------------------- -------------------- 4.7/9.5 MB 3.2 MB/s eta 0:00:02
   -------------------- ------------------- 5.0/9.5 MB 3.0 MB/s eta 0:00:02
   ---------------------- ----------------- 5.2/9.5 MB 2.9 MB/s eta 0:00:02
   ------------------------- -------------- 6.0/9.5 MB 2.8 MB/s eta 0:00:02
   --------------------------- ------------ 6.6/9.5 MB 2.9 MB/s eta 0:00:02
   ----------------------------- ---------- 7.1/9.5 MB 2.9 MB/s eta 0:00:01
   ----------------

In [9]:
pip install shap matplotlib

Note: you may need to restart the kernel to use updated packages.
